# Sprint 3 optional implementation companion

Complete the guided registry, replay and monitoring commands first. This notebook exposes the actual MLflow, PyTorch, NumPy, Pillow and scikit-learn operations behind the observations. It does not register, select or roll back models automatically. Use your own local records, not the author reference IDs. Keep independent attempts before opening feedback.

The Campus lesson and README are the primary execution route. The notebook adds closer inspection of native objects and raw data.

In [ ]:
from pathlib import Path
import os, csv, json, hashlib, io, zipfile
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score
import mlflow
from mlflow import MlflowClient
ROOT = Path.cwd()
if not (ROOT / "src/catalog_mlops").is_dir():
    ROOT = ROOT.parent
os.chdir(ROOT)
assert (ROOT / "work/replay/verification.json").is_file(), "Complete the guided replay and registry work first"
REGISTERED_NAME = "threadline-catalog"
uri = "sqlite:///" + (ROOT / "work/mlflow/tracking.db").as_posix()
mlflow.set_tracking_uri(uri)
mlflow.set_registry_uri(uri)
client = MlflowClient(tracking_uri=uri, registry_uri=uri)
CLASS_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

## From "which file is production?" to a registry

Read the selected alias once. A native model version identifies its source run and review tags. This is the current selection; a process started earlier can hold another version.

In [ ]:
version = client.get_model_version_by_alias(REGISTERED_NAME, "active")
print("Selected", version.name, version.version, "source run", version.run_id)
print("Native model address", version.source)
print("Validation tags", version.tags)
assert version.tags.get("validation_status") == "passed", "Finish recovery or validation before inspecting the approved model"

## Registering and promoting a model

Load that exact native version with PyTorch and predict one real thumbnail using visible fixed scaling. The image's known category is not an input to the model. Loading the version here pins the notebook's model even if the alias changes later.

In [ ]:
model = mlflow.pytorch.load_model(f"models:/{REGISTERED_NAME}/{version.version}", map_location="cpu")
model.eval()
with (ROOT / "work/cohorts/current/manifest.csv").open() as stream:
    first = next(csv.DictReader(stream))
content = (ROOT / "work/cohorts/current" / first["path"]).read_bytes()
with Image.open(io.BytesIO(content)) as image:
    assert image.mode == "L" and image.size == (28, 28)
    pixels = np.asarray(image).copy()
inputs = torch.from_numpy(pixels[None, None]).to(dtype=torch.float32) / 255.0
with torch.inference_mode():
    scores = model(inputs).softmax(dim=1)[0]
predicted = int(scores.argmax())
print(first["image_id"], CLASS_NAMES[predicted], float(scores[predicted]))

Each incoming thumbnail has one mean-brightness value. Reconstruct these values directly from PNG bytes, checking their fingerprints against the actual batch record. The function below is defined here so the file reading, hash check and NumPy calculation remain visible.

In [ ]:
def consumed_means(cohort):
    directory = ROOT / "work/cohorts" / cohort
    batch = json.loads((ROOT / "work/batches" / cohort / "batch.json").read_text())
    expected = {row["image_id"]: row["sha256"] for row in batch["inputs"]}
    with (directory / "manifest.csv").open() as stream:
        rows = list(csv.DictReader(stream))
    values = []
    for row in rows:
        content = (directory / row["path"]).read_bytes()
        assert hashlib.sha256(content).hexdigest() == expected[row["image_id"]], "Input changed after prediction"
        with Image.open(io.BytesIO(content)) as image:
            values.append(np.asarray(image).mean() / 255.0)
    return np.asarray(values)

brightness = {name: consumed_means(name) for name in ["reference", "current", "shifted-current"]}
fig, ax = plt.subplots(figsize=(9, 4.4), layout="constrained")
for name, color, style in [("reference", "#89998E", "-"), ("current", "#17482E", "--"), ("shifted-current", "#B91C1C", ":")]:
    ax.hist(brightness[name], bins=np.linspace(0, 1, 11), histtype="step", linewidth=2, color=color, linestyle=style, label=name)
ax.set(xlim=(0, 1), xlabel="Mean brightness per thumbnail (0–1)", ylabel="Number of thumbnails", title="Actual consumed image groups; shifted-current is constructed")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.show()

Known labels permit a different calculation: classification quality. Compute it from actual prediction rows and the separate label file, without a course scoring helper. Reference/current differences describe different image groups; ordinary current and its darkened copies share source identities but have different pixels.

In [ ]:
observed_quality = {}
for cohort in ["reference", "current", "shifted-current"]:
    with (ROOT / "work/batches" / cohort / "predictions.csv").open() as stream:
        predictions = list(csv.DictReader(stream))
    with (ROOT / "work/cohorts" / cohort / "labels.csv").open() as stream:
        labels = {row["image_id"]: int(row["label"]) for row in csv.DictReader(stream)}
    truth = np.array([labels[row["image_id"]] for row in predictions])
    guesses = np.array([int(row["class_id"]) for row in predictions])
    assert len(labels) == len(predictions) == 1000
    observed_quality[cohort] = {"accuracy": accuracy_score(truth, guesses), "macro_f1": f1_score(truth, guesses, labels=np.arange(10), average="macro", zero_division=0)}
print(observed_quality)

## What full reproducibility requires

Trace the selected native model back to its run and archived files. Inspect the actual source bytes against their recorded per-file fingerprints. A fingerprint is a comparison tool; the archive supplies the retrievable code.

In [ ]:
run = client.get_run(version.run_id)
record = json.loads((ROOT / "work/replay/recipe/run.json").read_text())
assert record["source"]["source_sha256"] == run.data.tags["source_sha256"]
assert record["split_sha256"] == run.data.tags["split_sha256"]
print("Parameters", run.data.params)
print("Environment", json.loads((ROOT / "work/replay/recipe/environment.json").read_text()))
with zipfile.ZipFile(ROOT / "work/replay/recipe/source.zip") as archive:
    observed = {name: hashlib.sha256(archive.read(name)).hexdigest() for name in archive.namelist()}
assert observed == record["source"]["file_sha256"]
verification = json.loads((ROOT / "work/replay/verification.json").read_text())
print("Declared policy", verification["policy"])
print("Observed checks", verification["checks"])

## Reproducing a past training run

Complete the independent past-run replay first. Inspect your new run record, fresh-environment observations and verification result. Record the actual differences and the deliberately altered evaluation's failed checks. Do not replace that work with the guided report.

In [ ]:
# Student work: inspect your independent replay evidence and explain its verdict.
# Keep a copied original run separate from a genuinely new training execution.
pass

<details><summary>Reference feedback — open after your attempt</summary>

**Result you should get**

Matching recipe identities and accepted numerical differences support the replay. A copied original execution or a manually increased metric must fail the relevant checks.

**How**

Read the original run, new execution record, actual environment/import paths and raw evaluation evidence together. Keep the declared policy fixed.

</details>

## Rolling back a bad model version

Complete the actual rollback in the terminal first. Compare live alias state with the model identities returned by the old and fresh HTTP processes. Explain what the reviewed journal captured and what the native incident record adds.

In [ ]:
# Student work: use retained actual registry, HTTP and batch observations.
# Explain why the old process and a fresh process can report different versions.
pass

<details><summary>Reference feedback — open after your attempt</summary>

**Result you should get**

The active alias and fresh consumer identify the approved target. The old consumer can remain pinned to the bad version until restarted. A successful HTTP status alone does not establish model quality or identity.

**How**

Inspect registered version, source run and checkpoint fingerprint; retain the actual reassignment and both process observations.

</details>

## Registry lifecycle deliverable

Use Morrow's original category requirements and your actual candidate records. Keep its eligibility decision separate from replay consistency and the aggregate quality alert. The handoff must include a real consumed incident and recovery, not just a prepared note.

In [ ]:
# Student work: inspect your selected Morrow run and its category metrics.
# Connect approval, active selection, incident, rollback and fresh loaded identity.
pass

<details><summary>Reference feedback — open after your attempt</summary>

**Result you should get**

An eligible model has connected replay/registry evidence and an observed recovery. The higher-average eight-epoch configuration can violate Morrow's T-shirt/top requirement.

**How**

Validate each decision against its own evidence: original business policy, declared replay checks and actual consumer identities. Keep raw records with the handoff.

</details>